In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Global Logistics Big Data Project") \
    .getOrCreate()


In [ ]:
# Load the Raw Dataset

In [2]:

df = spark.read.csv(
    "logiscale bigdata set.csv",
    header=True,
    inferSchema=True
)

df.show(5)


+-----------+--------+------------+-----------+--------------+------------+----------------+
|shipment_id|route_id|warehouse_id|distance_km|avg_speed_kmph|vehicle_type|  departure_date|
+-----------+--------+------------+-----------+--------------+------------+----------------+
|          1|    1051|           6|       1500|            42|         Van|01-01-2023 00:00|
|          2|    1092|          18|       1875|            69|       Truck|01-01-2023 00:01|
|          3|    1014|          18|       2831|            62|         Van|01-01-2023 00:02|
|          4|    1071|           7|       2253|            47|         Van|01-01-2023 00:03|
|          5|    1060|          14|       1675|            43|   Container|01-01-2023 00:04|
+-----------+--------+------------+-----------+--------------+------------+----------------+
only showing top 5 rows



In [3]:
df.count()

300000

In [4]:
df.printSchema()

root
 |-- shipment_id: integer (nullable = true)
 |-- route_id: integer (nullable = true)
 |-- warehouse_id: integer (nullable = true)
 |-- distance_km: integer (nullable = true)
 |-- avg_speed_kmph: integer (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- departure_date: string (nullable = true)



In [ ]:
# date value convertion

In [5]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn(
    "departure_timestamp",
    to_timestamp(col("departure_date"), "dd-MM-yyyy HH:mm")
)

In [6]:
df.select("departure_date","departure_timestamp").show(5)

+----------------+-------------------+
|  departure_date|departure_timestamp|
+----------------+-------------------+
|01-01-2023 00:00|2023-01-01 00:00:00|
|01-01-2023 00:01|2023-01-01 00:01:00|
|01-01-2023 00:02|2023-01-01 00:02:00|
|01-01-2023 00:03|2023-01-01 00:03:00|
|01-01-2023 00:04|2023-01-01 00:04:00|
+----------------+-------------------+
only showing top 5 rows



In [8]:
from pyspark.sql.functions import to_date

df = df.withColumn(
    "shipment_day",
    to_date("departure_timestamp"))

In [9]:
df.select("shipment_day").show(5)

+------------+
|shipment_day|
+------------+
|  2023-01-01|
|  2023-01-01|
|  2023-01-01|
|  2023-01-01|
|  2023-01-01|
+------------+
only showing top 5 rows



In [10]:
# ETA Calculation - ETA = distance / speed

In [10]:
from pyspark.sql.functions import col

df = df.withColumn(
    "eta_days",
    col("distance_km") / (col("avg_speed_kmph") * 24)
)

In [11]:
df.select("distance_km","avg_speed_kmph","eta_days").show(10)

+-----------+--------------+-------------------+
|distance_km|avg_speed_kmph|           eta_days|
+-----------+--------------+-------------------+
|       1500|            42| 1.4880952380952381|
|       1875|            69| 1.1322463768115942|
|       2831|            62| 1.9025537634408602|
|       2253|            47|  1.997340425531915|
|       1675|            43|  1.623062015503876|
|       1761|            43| 1.7063953488372092|
|        477|            46| 0.4320652173913043|
|       2129|            72| 1.2320601851851851|
|        329|            68|0.20159313725490197|
|        313|            78| 0.1672008547008547|
+-----------+--------------+-------------------+
only showing top 10 rows



In [ ]:
# ATA Calculation

In [12]:
from pyspark.sql.functions import rand

df = df.withColumn(
    "ata_days",
    col("eta_days") + (rand()*2)
)

In [13]:
df.select("eta_days","ata_days").show(10)

+-------------------+------------------+
|           eta_days|          ata_days|
+-------------------+------------------+
| 1.4880952380952381|2.4879483129852487|
| 1.1322463768115942|  1.77280575477884|
| 1.9025537634408602|1.9066822179269154|
|  1.997340425531915| 3.268279889591386|
|  1.623062015503876| 1.654956468839204|
| 1.7063953488372092| 2.806519716635649|
| 0.4320652173913043|0.9762415714560803|
| 1.2320601851851851|3.1145669681768484|
|0.20159313725490197| 2.002477571519213|
| 0.1672008547008547|0.2801628735334306|
+-------------------+------------------+
only showing top 10 rows



In [15]:
# Calculate Delivery Delay

In [14]:
df = df.withColumn(
    "delay_days",
    col("ata_days") - col("eta_days")
)

In [15]:
df.select("eta_days","ata_days","delay_days").show(10)

+-------------------+------------------+--------------------+
|           eta_days|          ata_days|          delay_days|
+-------------------+------------------+--------------------+
| 1.4880952380952381|2.4879483129852487|  0.9998530748900105|
| 1.1322463768115942|  1.77280575477884|  0.6405593779672458|
| 1.9025537634408602|1.9066822179269154|0.004128454486055144|
|  1.997340425531915| 3.268279889591386|  1.2709394640594711|
|  1.623062015503876| 1.654956468839204| 0.03189445333532803|
| 1.7063953488372092| 2.806519716635649|  1.1001243677984396|
| 0.4320652173913043|0.9762415714560803|  0.5441763540647759|
| 1.2320601851851851|3.1145669681768484|  1.8825067829916633|
|0.20159313725490197| 2.002477571519213|   1.800884434264311|
| 0.1672008547008547|0.2801628735334306| 0.11296201883257592|
+-------------------+------------------+--------------------+
only showing top 10 rows



In [18]:
#Route Efficiency Analysis

In [16]:
route_delay = df.groupBy("route_id") \
                .avg("delay_days") \
                .withColumnRenamed("avg(delay_days)","avg_delay")

route_delay.show()

+--------+------------------+
|route_id|         avg_delay|
+--------+------------------+
|    1088|  1.00867434040388|
|    1084|1.0050188644380178|
|    1025|1.0118425717093364|
|    1005|0.9875767129293426|
|    1016| 1.014276638203689|
|    1068|1.0062436805136497|
|    1031|1.0047181924102206|
|    1051|  1.00634609861069|
|    1034|0.9725781119743419|
|    1064| 0.983157344623672|
|    1030|1.0125871923296899|
|    1019|1.0076675060370985|
|    1085|  1.00116929420596|
|    1056|1.0105484635342854|
|    1046|0.9915792867135803|
|    1093|0.9885518171833443|
|    1077| 0.987038090673444|
|    1065|1.0013250649825824|
|    1061|1.0007549783502756|
|    1055|1.0096368540946752|
+--------+------------------+
only showing top 20 rows



In [21]:
# Warehouse Shipment Volume

In [17]:
warehouse_volume = df.groupBy("warehouse_id").count()

warehouse_volume.show()

+------------+-----+
|warehouse_id|count|
+------------+-----+
|          12|11769|
|          22|12103|
|           1|11980|
|          13|12064|
|           6|11900|
|          16|12152|
|           3|12043|
|          20|11989|
|           5|12065|
|          19|12241|
|          15|12245|
|           9|12021|
|          17|12051|
|           4|12026|
|           8|12106|
|          23|12021|
|           7|11873|
|          10|11985|
|          25|11828|
|          24|12004|
+------------+-----+
only showing top 20 rows



In [23]:
# Distance vs Delay Analysis

In [18]:
distance_delay = df.groupBy("distance_km") \
                   .avg("delay_days") \
                   .withColumnRenamed("avg(delay_days)","avg_delay")

distance_delay.show()


+-----------+------------------+
|distance_km|         avg_delay|
+-----------+------------------+
|        148|0.9353335958951533|
|        496|1.1324355284440153|
|        471|0.9824851737105346|
|       2659|1.0174418148128388|
|       1829|0.9503647047739928|
|       2142|0.9424622665908703|
|       1088|0.9376220061321208|
|       2366|1.0776472073733419|
|       2866| 1.034215370806132|
|       1342|0.9600102716233434|
|       1591|0.9425427877013448|
|       1580|0.8802327240320184|
|       1645| 0.957945802255862|
|        463|1.0903348024789932|
|        833|0.9618722864913555|
|       1959|0.9394201793997949|
|       2122|0.9788739906387106|
|       1238|1.1002368190633023|
|       1025|0.9894657779658926|
|       2811| 0.951123517402793|
+-----------+------------------+
only showing top 20 rows



In [30]:
# Moving Average Forecast

In [21]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

windowSpec = Window.orderBy("shipment_day").rowsBetween(-3,0)

forecast = daily_shipments.withColumn(
    "moving_avg",
    avg("count").over(windowSpec)
)

forecast.show()

+------------+-----+----------+
|shipment_day|count|moving_avg|
+------------+-----+----------+
|  2023-01-01| 1440|    1440.0|
|  2023-01-02| 1440|    1440.0|
|  2023-01-03| 1440|    1440.0|
|  2023-01-04| 1440|    1440.0|
|  2023-01-05| 1440|    1440.0|
|  2023-01-06| 1440|    1440.0|
|  2023-01-07| 1440|    1440.0|
|  2023-01-08| 1440|    1440.0|
|  2023-01-09| 1440|    1440.0|
|  2023-01-10| 1440|    1440.0|
|  2023-01-11| 1440|    1440.0|
|  2023-01-12| 1440|    1440.0|
|  2023-01-13| 1440|    1440.0|
|  2023-01-14| 1440|    1440.0|
|  2023-01-15| 1440|    1440.0|
|  2023-01-16| 1440|    1440.0|
|  2023-01-17| 1440|    1440.0|
|  2023-01-18| 1440|    1440.0|
|  2023-01-19| 1440|    1440.0|
|  2023-01-20| 1440|    1440.0|
+------------+-----+----------+
only showing top 20 rows



In [22]:
# Export Results for Power BI

In [23]:
route_delay.toPandas().to_csv("route_delay_summary.csv", index=False)

warehouse_volume.toPandas().to_csv("warehouse_volume.csv", index=False)

daily_shipments.toPandas().to_csv("shipment_trend.csv", index=False)

In [24]:
import os
os.listdir()

['.ipynb_checkpoints',
 '.~logiscale bigdata set.csv',
 'abhi pro 123.sql',
 'Abhinav vm -P19MT23M015033  (1).docx',
 'Abhinav vm . P19MT23M015033.docx',
 'Abhinav vm . P19MT23M015033.pdf',
 'Advanced_Excel_Practice_File.xlsx',
 'Amazon Store Sales Data.xlsx - Sheet1 (1).csv',
 'amazon_products_sales_data_cleaned.csv',
 'amazon_products_sales_data_uncleaned.csv',
 'Anaconda3-2025.12-2-Windows-x86_64.exe',
 'archive.zip',
 'cleaned_dataset_python (1).ipynb',
 'cleaned_dataset_retail-project.csv',
 'cleaned_python.csv',
 'combined (1).xlsx',
 'combined.xlsx',
 'combined_dashboard.png',
 'consumer360_rfm_customer_segment.csv',
 'consumer360_rfm_customer_segment.ipynb',
 'consumer360_rfm_customer_segment.py',
 'Country-Code.xlsx',
 'Customer 360 – Segmentation & RFM Analysis (1).pbix',
 'Customer 360 – Segmentation & RFM Analysis.pbix',
 'data analylitics project-Employee_attendence.xlsx',
 'Data analyst Resume.pdf (1).pdf',
 'Data analyst Resume.pdf (2).pdf',
 'Data analyst Resume.pdf.pdf